In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, get_token_bilstm_embeddings_from_string_with_model, get_adapter_embeddings_from_string_with_model, make_graph_ready_for_token_ids
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
from eval_utils import ensure_in_seq_string_form, cos

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [21]:
# s1 = ['G:7', 'C:maj']
# s2 = ['G:maj', 'C:maj7']
# s3 = ['C#:7', 'C:maj7']

# s1 = ['G:7', 'C:maj']
# s2 = ['G:maj', 'A:min7']
# s3 = ['E:7', 'A:min7']

# s1 = ['A:7', 'C:7']
# s2 = ['A:7', 'D:7', 'C:7']
# s3 = ['A:7', 'D:7', 'G:7', 'C:7']

# s1 = ['G:7', 'C:7']
# s2 = ['A:7', 'D:7']
# s3 = ['F:7', 'A#:7']

# 'b_A:min_@2;C#:maj_@2',
# s1 = ['A:min', 'C#:maj']
# s2 = ['A:min7', 'C#:maj6']
# s3 = ['A#:min', 'D:maj']

# 'b_G:maj_@2;A#:11_@2',
s1 = ['G:maj', 'A#:11']
s2 = ['G:maj6', 'A#:7']
s3 = ['G#:maj', 'B:11']

# s1 = ['E:maj13', 'G#:sus2']
# s2 = ['E:maj', 'G#:9']
# s3 = ['F:maj', 'A:9']

# s1 = ['D#:minmaj7', 'B:maj6']
# s2 = ['D#:min', 'B:maj']
# s3 = ['D:min', 'C:maj']

In [22]:
st1 = ensure_in_seq_string_form(s1)
st2 = ensure_in_seq_string_form(s2)
st3 = ensure_in_seq_string_form(s3)

print(st1)
print(st2)
print(st3)

b_G:maj_@2A#:11_@2
b_G:maj6_@2A#:7_@2
b_G#:maj_@2B:11_@2


In [23]:
y_graph_s1 = get_graph_embeddings_from_string_with_model(st1, graph_adapter_model)
y_graph_s2 = get_graph_embeddings_from_string_with_model(st2, graph_adapter_model)
y_graph_s3 = get_graph_embeddings_from_string_with_model(st3, graph_adapter_model)

y_token_s1 = get_token_bilstm_embeddings_from_string_with_model(st1, token_adapter_model)
y_token_s2 = get_token_bilstm_embeddings_from_string_with_model(st2, token_adapter_model)
y_token_s3 = get_token_bilstm_embeddings_from_string_with_model(st3, token_adapter_model)

y_adapter_s1 = get_adapter_embeddings_from_string_with_model(st1, adapter_model, graph_adapter_model, token_adapter_model)
y_adapter_s2 = get_adapter_embeddings_from_string_with_model(st2, adapter_model, graph_adapter_model, token_adapter_model)
y_adapter_s3 = get_adapter_embeddings_from_string_with_model(st3, adapter_model, graph_adapter_model, token_adapter_model)

In [24]:
print(f's1: {s1} | s2: {s2} | s3: {s3}')

print('graph st1-st2: ', cos(y_graph_s1, y_graph_s2))
print('graph st2-st3: ', cos(y_graph_s2, y_graph_s3))
print('graph st1-st3: ', cos(y_graph_s1, y_graph_s3))

print('token st1-st2: ', cos(y_token_s1, y_token_s2))
print('token st2-st3: ', cos(y_token_s2, y_token_s3))
print('token st1-st3: ', cos(y_token_s1, y_token_s3))

print('adapter st1-st2: ', cos(y_adapter_s1, y_adapter_s2))
print('adapter st2-st3: ', cos(y_adapter_s2, y_adapter_s3))
print('adapter st1-st3: ', cos(y_adapter_s1, y_adapter_s3))

s1: ['G:maj', 'A#:11'] | s2: ['G:maj6', 'A#:7'] | s3: ['G#:maj', 'B:11']
graph st1-st2:  tensor([0.5368], device='cuda:2')
graph st2-st3:  tensor([0.2896], device='cuda:2')
graph st1-st3:  tensor([0.1554], device='cuda:2')
token st1-st2:  tensor([0.1841], device='cuda:2')
token st2-st3:  tensor([0.3817], device='cuda:2')
token st1-st3:  tensor([0.0905], device='cuda:2')
adapter st1-st2:  tensor([0.5270], device='cuda:2')
adapter st2-st3:  tensor([0.3454], device='cuda:2')
adapter st1-st3:  tensor([0.1676], device='cuda:2')
